# Тарифы по ОГРН (клиенты с фото)

Минимальная тетрадка для проверки:
1. Импорты
2. Подключение к Impala
3. Список клиентов (ОГРН)
4. Поиск `tariff_name` по OGRN через R2-цепочку

In [ ]:
import re

import pandas as pd
from rail_connectors.connection import connect

pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', None)
pd.set_option('display.width', 240)

In [ ]:
imp = connect(
    to='IMPALA',
    extra_options={'db': 'sandbox_ai'},
    driver_args={'tez.queue.name': 'ai'},
    kerberos={
        'keytab_path': '/home/jovyan/test_requests/tech.keytab',
        'use_credentials': True,
        'update_keytab': True,
    },
    user_params={'user_name': 'Shestopalov-VYur'}
)
imp._init_connection()

## Список клиентов (из фото)

In [ ]:
clients_from_photo = [
    {'client_name_photo': 'БИКМЕТОВА САЗИДА АСХАТОВНА (ИП)', 'contract_number_photo': 'ЭД266219/00028', 'ogrn_photo': '325028000279052'},
    {'client_name_photo': 'ЛАРИН АЛЕКСАНДР НИКОЛАЕВИЧ (ИП ГКФХ)', 'contract_number_photo': 'ЭД266213/00034', 'ogrn_photo': '304025512800036'},
    {'client_name_photo': 'БАГАУТДИНОВ РАБИЛ ЗИЯФОВИЧ (ИП)', 'contract_number_photo': 'ЭД266206/00045', 'ogrn_photo': '326028000007223'},
    {'client_name_photo': 'ЛИ ЖЭНЬШУНЬ (ИП)', 'contract_number_photo': 'ЭД266235/00033', 'ogrn_photo': '320028000097857'},
    {'client_name_photo': 'ООО "НЬЮРИС"', 'contract_number_photo': 'ВД_300', 'ogrn_photo': '1149102120441'},
    {'client_name_photo': 'ФРАНТЕНКО ВИКТОРИЯ КОНСТАНТИНОВНА (ИП)', 'contract_number_photo': 'ЭД256615/00027', 'ogrn_photo': '323385000127952'},
    {'client_name_photo': 'ООО "КРОСС+"', 'contract_number_photo': 'ВД_000098', 'ogrn_photo': '1053811141230'},
    {'client_name_photo': 'ООО "САНИВА"', 'contract_number_photo': 'ЭД263226/00020', 'ogrn_photo': '1114613000490'},
    {'client_name_photo': 'ООО "СВТ"', 'contract_number_photo': 'ЭД250800/00022', 'ogrn_photo': '1095321000070'},
    {'client_name_photo': 'ООО "КУПЕЦ"', 'contract_number_photo': 'ЭД262553/00005', 'ogrn_photo': '1105543001520'},
    {'client_name_photo': 'КИЯН МАРИНА НИКОЛАЕВНА (ИП)', 'contract_number_photo': 'ЭД262504/00044', 'ogrn_photo': '326547600049591'},
    {'client_name_photo': 'ЛАНГ ЭДУАРД ЭДУАРДОВИЧ (ИП)', 'contract_number_photo': 'ЭД262500/00043', 'ogrn_photo': '322547600045721'},
    {'client_name_photo': 'МУШЕНКО МАКСИМ АНАТОЛЬЕВИЧ (ИП)', 'contract_number_photo': 'ЭД252500/00007', 'ogrn_photo': '323547600047523'},
    {'client_name_photo': 'АЛЛАЗОВА СУДАБА ИЛЬЯС КЫЗЫ (ИП)', 'contract_number_photo': 'ЭД252500/00010', 'ogrn_photo': '324246800091492'},
    {'client_name_photo': 'ООО "ПРИВОЛЬНОЕ"', 'contract_number_photo': 'ЭД260519/00077', 'ogrn_photo': '1115658002020'},
    {'client_name_photo': 'МИНАЗКИРОВ КИРИЛЛ ЭДУАРДОВИЧ (ИП)', 'contract_number_photo': 'ЭД250507/00108', 'ogrn_photo': '325565800030598'},
    {'client_name_photo': 'ООО "САНФОРТ-СЕРВИС"', 'contract_number_photo': 'ЭД261500/00015', 'ogrn_photo': '1025801369076'},
    {'client_name_photo': 'ТУГАЕВА ЮЛИЯ ВАЛЕРЬЕВНА (ИП)', 'contract_number_photo': 'ЭД261500/00016', 'ogrn_photo': '307583611500025'},
    {'client_name_photo': 'ШАБАЛИН ДМИТРИЙ НИКОЛАЕВИЧ (ИП ГКФХ)', 'contract_number_photo': 'ЭД267605/00036', 'ogrn_photo': '325595800099432'},
    {'client_name_photo': 'ООО "ЛИОНА"', 'contract_number_photo': 'ЭД267605/00043', 'ogrn_photo': '1225900023315'},
    {'client_name_photo': 'СЕЛЕЗНЕВА МАРГАРИТА ФАНИЛЬЕВНА (ИП)', 'contract_number_photo': 'ЭД267309/00035', 'ogrn_photo': '326965800023753'},
    {'client_name_photo': 'ПОПОВ АЛЕКСАНДР ВЛАДИМИРОВИЧ (ИП ГКФХ)', 'contract_number_photo': 'ЭД267300/00047', 'ogrn_photo': '319665800113563'},
    {'client_name_photo': 'НЕСТЕРОВА МАРИНА АНДРЕЕВНА (ИП)', 'contract_number_photo': 'ЭД267306/00038', 'ogrn_photo': '322665800105220'},
    {'client_name_photo': 'ЗИННАТУЛЛИНА ГУЗАЛЬ ВАКЫФОВНА (ИП ГКФХ)', 'contract_number_photo': 'ЭД266700/00045', 'ogrn_photo': '312167520800136'},
    {'client_name_photo': 'ГИЗЕТДИНОВ ИЛЬНУР АЛЬБЕРТОВИЧ (ИП)', 'contract_number_photo': 'ЭД266721/00037', 'ogrn_photo': '326169000072080'},
    {'client_name_photo': 'СИБГАТУЛЛИНА АЛИЯ ИРЕКОВНА (ИП)', 'contract_number_photo': 'ЭД256705/00004', 'ogrn_photo': '320169000141554'},
    {'client_name_photo': 'ООО "АВТОДЕТАЛИ"', 'contract_number_photo': 'ЭД266716/00040', 'ogrn_photo': '1131651000250'},
    {'client_name_photo': 'АШИМЖОНОВ РАСУЛЖОН АРАБЖОНОВИЧ (ИП)', 'contract_number_photo': 'ЭД256705/00005', 'ogrn_photo': '313169000166854'},
]

clients_df = pd.DataFrame(clients_from_photo)
clients_df.insert(0, 'photo_row_id', range(1, len(clients_df) + 1))

clients_df['ogrn'] = (
    clients_df['ogrn_photo']
    .astype(str)
    .str.replace(r'\D+', '', regex=True)
)

# Нормализуем номер договора для точной привязки тарифа к клиенту
clients_df['contract_number_norm'] = (
    clients_df['contract_number_photo']
    .astype(str)
    .str.upper()
    .str.replace('Ё', 'Е', regex=False)
    .str.replace(r'[^A-ZА-Я0-9]+', '', regex=True)
)
clients_df['contract_number_norm'] = clients_df['contract_number_norm'].str.replace(r'^(?:ВД|ЭДЭ?)', 'ЭД', regex=True)

clients_df = clients_df[clients_df['ogrn'].str.len() > 0].copy()
ogrn_values = sorted(clients_df['ogrn'].dropna().astype(str).unique().tolist())

print(f'Клиентов в списке: {len(clients_df):,}')
print(f'Уникальных ОГРН: {len(ogrn_values):,}')
clients_df[['photo_row_id', 'client_name_photo', 'contract_number_photo', 'ogrn']].head(30)

## Поиск tariff_name по OGRN (R2-цепочка)

In [ ]:
if not ogrn_values:
    raise ValueError('Список OGRN пуст. Проверь входные данные.')

ogrn_sql = ', '.join([f"'{x}'" for x in ogrn_values])

sql_ogrn_tariff = f"""
with corp_in as (
    select
        regexp_replace(trim(cast(c.c_register_gos_reg_num_rec as string)), '[^0-9]', '') as ogrn,
        cast(c.id as string) as cft_id
    from ods.scd1_z_cl_corp c
    where regexp_replace(trim(cast(c.c_register_gos_reg_num_rec as string)), '[^0-9]', '') in ({ogrn_sql})
),
agr_one as (
    select
        cast(a.abs_agr_id as string) as agr_id,
        cast(a.c_agr_number as string) as contract_number_alpha,
        row_number() over (
            partition by cast(a.abs_agr_id as string)
            order by cast(a.d_valid_from as date) desc, cast(a.n_agr as string) desc
        ) as rn
    from ods_alpha.scd1_agreements a
    where coalesce(a.ods_deleted_flg, '0') <> '1'
),
m as (
    select
        cast(m.c_cl_org as string) as cft_id,
        cast(m.id as string) as agr_id,
        cast(m.c_tariff_plan as string) as c_tariff_plan
    from ods.scd1_z_r2_ip_merchants m
)
select
    ci.ogrn,
    m.agr_id,
    a.contract_number_alpha,
    m.c_tariff_plan,
    cast(tp.c_name as string) as tariff_name
from corp_in ci
left join m
    on m.cft_id = ci.cft_id
left join ods.scd1_z_r2_tariff_plan tp
    on cast(tp.id as string) = m.c_tariff_plan
left join agr_one a
    on a.agr_id = m.agr_id
   and a.rn = 1
"""

with imp:
    imp.execute('set MEM_LIMIT=8g')
    ogrn_tariff_raw_df = imp.fetch(sql_ogrn_tariff)

if ogrn_tariff_raw_df is None:
    ogrn_tariff_raw_df = pd.DataFrame(columns=['ogrn', 'agr_id', 'contract_number_alpha', 'c_tariff_plan', 'tariff_name'])

for c in ['ogrn', 'agr_id', 'contract_number_alpha', 'c_tariff_plan', 'tariff_name']:
    if c in ogrn_tariff_raw_df.columns:
        ogrn_tariff_raw_df[c] = ogrn_tariff_raw_df[c].astype(str).str.strip()

# Нормализуем номер договора из Alpha для привязки
ogrn_tariff_raw_df['contract_number_alpha_norm'] = (
    ogrn_tariff_raw_df['contract_number_alpha']
    .astype(str)
    .str.upper()
    .str.replace('Ё', 'Е', regex=False)
    .str.replace(r'[^A-ZА-Я0-9]+', '', regex=True)
)
ogrn_tariff_raw_df['contract_number_alpha_norm'] = ogrn_tariff_raw_df['contract_number_alpha_norm'].str.replace(r'^(?:ВД|ЭДЭ?)', 'ЭД', regex=True)

# Полный список связок OGRN -> agr_id -> tariff_name
ogrn_tariff_full_df = clients_df.merge(
    ogrn_tariff_raw_df,
    on='ogrn',
    how='left'
).sort_values(
    ['ogrn', 'agr_id', 'tariff_name'],
    ascending=[True, True, True]
).reset_index(drop=True)

# Флаг совпадения номера договора (основная логика привязки тарифа)
ogrn_tariff_full_df['is_contract_match'] = (
    ogrn_tariff_full_df['contract_number_norm'].notna()
    & ogrn_tariff_full_df['contract_number_alpha_norm'].notna()
    & (ogrn_tariff_full_df['contract_number_norm'] == ogrn_tariff_full_df['contract_number_alpha_norm'])
)

# Итоговая привязка тарифа к клиенту: 1 строка на клиента по номеру договора
ogrn_tariff_bound_by_contract_df = (
    ogrn_tariff_full_df[ogrn_tariff_full_df['is_contract_match']]
    .sort_values(['photo_row_id', 'agr_id'])
    .drop_duplicates(subset=['photo_row_id'], keep='first')
    .reset_index(drop=True)
)

final_tariff_by_contract_df = clients_df.merge(
    ogrn_tariff_bound_by_contract_df[[
        'photo_row_id', 'agr_id', 'contract_number_alpha', 'c_tariff_plan', 'tariff_name'
    ]],
    on='photo_row_id',
    how='left'
)

# QC
input_ogrn_cnt = len(set(ogrn_values))
found_ogrn_cnt = int(
    ogrn_tariff_full_df.dropna(subset=['agr_id']).groupby('ogrn')['agr_id'].nunique().gt(0).sum()
)
missing_ogrn_df = (
    clients_df[['ogrn']].drop_duplicates()
    .merge(
        ogrn_tariff_full_df[['ogrn', 'agr_id']].dropna(subset=['agr_id']).drop_duplicates(subset=['ogrn']),
        on='ogrn',
        how='left',
        indicator=True
    )
)
missing_ogrn_df = missing_ogrn_df[missing_ogrn_df['_merge'] == 'left_only'][['ogrn']].reset_index(drop=True)

multi_agr_by_ogrn_df = (
    ogrn_tariff_full_df.dropna(subset=['agr_id'])
    .groupby('ogrn', as_index=False)
    .agg(agr_cnt=('agr_id', 'nunique'))
)
multi_agr_by_ogrn_df = multi_agr_by_ogrn_df[multi_agr_by_ogrn_df['agr_cnt'] > 1].sort_values('agr_cnt', ascending=False)

bound_clients_cnt = int(final_tariff_by_contract_df['tariff_name'].notna().sum())
not_bound_clients_df = final_tariff_by_contract_df[final_tariff_by_contract_df['tariff_name'].isna()].copy()

contract_multi_match_df = (
    ogrn_tariff_full_df[ogrn_tariff_full_df['is_contract_match']]
    .groupby('photo_row_id', as_index=False)
    .agg(match_cnt=('agr_id', 'nunique'))
)
contract_multi_match_df = contract_multi_match_df[contract_multi_match_df['match_cnt'] > 1]

print('QC по подтяжке tariff_name по OGRN:')
print(f'Уникальных ОГРН во входе: {input_ogrn_cnt:,}')
print(f'ОГРН, где найден хотя бы один agr_id: {found_ogrn_cnt:,}')
print(f'ОГРН без найденного agr_id: {len(missing_ogrn_df):,}')
print(f'ОГРН с несколькими agr_id: {len(multi_agr_by_ogrn_df):,}')
print(f'Клиентов, привязанных к тарифу по номеру договора: {bound_clients_cnt:,} из {len(clients_df):,}')
print(f'Клиентов с несколькими совпадениями по номеру договора: {len(contract_multi_match_df):,}')

print('\nПолный список связок OGRN -> agr_id -> tariff_name:')
display(ogrn_tariff_full_df)

print('\nИтоговая привязка тарифа к клиентам по номеру договора:')
display(final_tariff_by_contract_df[[
    'photo_row_id', 'client_name_photo', 'contract_number_photo', 'ogrn',
    'agr_id', 'contract_number_alpha', 'c_tariff_plan', 'tariff_name'
]])

if not missing_ogrn_df.empty:
    print('\nОГРН без найденного agr_id:')
    display(missing_ogrn_df)

if not multi_agr_by_ogrn_df.empty:
    print('\nОГРН, у которых несколько agr_id:')
    display(multi_agr_by_ogrn_df)

if not not_bound_clients_df.empty:
    print('\nКлиенты, которых не удалось привязать к тарифу по номеру договора:')
    display(not_bound_clients_df[['photo_row_id', 'client_name_photo', 'contract_number_photo', 'ogrn']])

if not contract_multi_match_df.empty:
    print('\nКлиенты с несколькими совпадениями по номеру договора (нужна ручная проверка):')
    display(contract_multi_match_df.merge(
        clients_df[['photo_row_id', 'client_name_photo', 'contract_number_photo', 'ogrn']],
        on='photo_row_id',
        how='left'
    ))

## Привязка тарифов к клиентам по номеру договора (отдельная секция)

Секция использует уже полученные результаты `clients_df` и `ogrn_tariff_raw_df`.
Результат: `final_tariff_by_contract_df` (1 строка = 1 клиент).

In [ ]:
if 'clients_df' not in globals() or 'ogrn_tariff_raw_df' not in globals():
    raise ValueError('Сначала запусти секции со списком клиентов и SQL (ячейки 4 и 6).')

bind_left_df = clients_df.copy()
bind_right_df = ogrn_tariff_raw_df.copy()

# Нормализация номера договора с обеих сторон
if 'contract_number_norm' not in bind_left_df.columns:
    bind_left_df['contract_number_norm'] = (
        bind_left_df['contract_number_photo']
        .astype(str)
        .str.upper()
        .str.replace('Ё', 'Е', regex=False)
        .str.replace(r'[^A-ZА-Я0-9]+', '', regex=True)
        .str.replace(r'^(?:ВД|ЭДЭ?)', 'ЭД', regex=True)
    )

bind_right_df['contract_number_alpha_norm'] = (
    bind_right_df['contract_number_alpha']
    .astype(str)
    .str.upper()
    .str.replace('Ё', 'Е', regex=False)
    .str.replace(r'[^A-ZА-Я0-9]+', '', regex=True)
    .str.replace(r'^(?:ВД|ЭДЭ?)', 'ЭД', regex=True)
)

bind_all_df = bind_left_df.merge(bind_right_df, on='ogrn', how='left')
bind_all_df['is_contract_match'] = (
    bind_all_df['contract_number_norm'].notna()
    & bind_all_df['contract_number_alpha_norm'].notna()
    & (bind_all_df['contract_number_norm'] == bind_all_df['contract_number_alpha_norm'])
)

bind_match_df = (
    bind_all_df[bind_all_df['is_contract_match']]
    .sort_values(['photo_row_id', 'agr_id'])
    .drop_duplicates(subset=['photo_row_id'], keep='first')
    .reset_index(drop=True)
)

final_tariff_by_contract_df = bind_left_df.merge(
    bind_match_df[[
        'photo_row_id', 'agr_id', 'contract_number_alpha', 'c_tariff_plan', 'tariff_name'
    ]],
    on='photo_row_id',
    how='left'
)

bound_clients_cnt = int(final_tariff_by_contract_df['tariff_name'].notna().sum())
not_bound_clients_df = final_tariff_by_contract_df[final_tariff_by_contract_df['tariff_name'].isna()].copy()

contract_multi_match_df = (
    bind_all_df[bind_all_df['is_contract_match']]
    .groupby('photo_row_id', as_index=False)
    .agg(match_cnt=('agr_id', 'nunique'))
)
contract_multi_match_df = contract_multi_match_df[contract_multi_match_df['match_cnt'] > 1]

print('QC по привязке тарифа к клиентам по номеру договора:')
print(f'Клиентов, привязанных к тарифу: {bound_clients_cnt:,} из {len(bind_left_df):,}')
print(f'Клиентов без привязки: {len(not_bound_clients_df):,}')
print(f'Клиентов с несколькими совпадениями: {len(contract_multi_match_df):,}')

print('\nИтоговая таблица (1 строка = 1 клиент):')
display(final_tariff_by_contract_df[[
    'photo_row_id', 'client_name_photo', 'contract_number_photo', 'ogrn',
    'agr_id', 'contract_number_alpha', 'c_tariff_plan', 'tariff_name'
]])

if not not_bound_clients_df.empty:
    print('\nНе удалось привязать по номеру договора:')
    display(not_bound_clients_df[['photo_row_id', 'client_name_photo', 'contract_number_photo', 'ogrn']])

if not contract_multi_match_df.empty:
    print('\nНесколько совпадений по номеру договора (нужна ручная проверка):')
    display(contract_multi_match_df.merge(
        bind_left_df[['photo_row_id', 'client_name_photo', 'contract_number_photo', 'ogrn']],
        on='photo_row_id',
        how='left'
    ))